# ChurnSense — Preprocessing & Feature Engineering

## Objective

The objective of this notebook is to prepare the Telco Customer Churn dataset for machine learning.

We will:
- Remove unnecessary identifier columns
- Convert the target variable into numerical form
- Separate numerical and categorical features
- Build a preprocessing pipeline
- Split the data using stratification
- Create engineered features

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [18]:
df = pd.read_csv(r"C:\Users\PARTH\Downloads\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [19]:
df = df.drop("customerID", axis=1)

In [20]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [21]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [22]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [24]:
X_train["tenure_group"] = pd.cut(
    X_train["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

X_test["tenure_group"] = pd.cut(
    X_test["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

In [25]:
categorical_features.append("tenure_group")

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [28]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [29]:
X_train_processed.shape

(5634, 49)

In [15]:
import sys
sys.path.append("..")

from src.preprocessing import load_data, prepare_data, create_preprocessor

In [16]:
df_test = load_data("../WA_Fn-UseC_-Telco-Customer-Churn.csv")

X_test_module, y_test_module, numeric_features_module, categorical_features_module = prepare_data(df_test)

print("X shape:", X_test_module.shape)
print("y shape:", y_test_module.shape)
print("Numeric features:", len(numeric_features_module))
print("Categorical features:", len(categorical_features_module))

X shape: (7043, 20)
y shape: (7043,)
Numeric features: 4
Categorical features: 16


In [17]:
preprocessor_module = create_preprocessor(
    numeric_features_module,
    categorical_features_module
)

X_processed_module = preprocessor_module.fit_transform(X_test_module)

print("Processed data shape:", X_processed_module.shape)

Processed data shape: (7043, 49)
